In [ ]:
import torch

from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "google/gemma-3-4b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float32,
    device_map="cpu",
)

messages = [
    {"role": "user", "content": "Hello! Can you introduce yourself?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)
outputs = model.generate(  # type: ignore
    **inputs,
    max_new_tokens=256,
    do_sample=False,  # temperature = 0.0
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)
print(response)

Loading weights:   0%|          | 1/883 [00:01<18:59,  1.29s/it]

: 

In [1]:
import json


class GemmaTokenizer:
    def __init__(self, vocab: dict[str, int], merges: list[list[str]]):
        self.vocab = vocab
        self.idx_to_str = {i: s for s, i in vocab.items()}

        self.ranks = {}
        self.merges = {}

        for i, (s1, s2) in enumerate(merges):
            pair = (vocab[s1], vocab[s2])
            self.ranks[pair] = i
            self.merges[pair] = vocab[s1 + s2]

    def encode(self, text: str) -> list[int]:
        text = text.replace(" ", "▁")

        tokens = [self.vocab["<bos>"]]
        for char in text:
            if char in self.vocab:
                tokens.append(self.vocab[char])
            else:
                for byte in char.encode("utf-8", errors="replace"):
                    tokens.append(self.vocab[f"<0x{hex(byte)[2:].upper()}>"])

        while True:
            best_rank = float("inf")
            best_pair = None

            for pair in zip(tokens[1:], tokens[2:]):
                if ((rank := self.ranks.get(pair)) is not None) and rank < best_rank:
                    best_rank = rank
                    best_pair = pair

            if best_pair is None:
                break

            i = 1
            while i < len(tokens) - 1:
                pair = (tokens[i], tokens[i + 1])
                if pair == best_pair:
                    tokens[i:i + 2] = [self.merges[pair]]
                i += 1

        return tokens

    def decode(self, tokens: list[int]) -> str:
        return (
            "".join(self.idx_to_str[tok] for tok in tokens)
        ).replace("▁", " ")


with open("./Models/gemma-3-1b-it/tokenizer.json", encoding="utf-8") as f:
    data = json.load(f)

tokenizer = GemmaTokenizer(data["model"]["vocab"], data["model"]["merges"])

In [ ]:
from export import load_bin

model, tokenizer = load_bin("exported/gemma-3-12b-it_q.bin")

In [ ]:
import torch
import torch.nn.functional as F

FP16_MAX = torch.finfo(torch.float16).max

from model import precompute_freqs_cis, apply_rotary_emb, GemmaDecoderBlock

config = model.config

freqs_cis = precompute_freqs_cis(config.head_dim, 1024, config.local_theta)

def check(name, t, i):
    if torch.isinf(t).any() or torch.isnan(t).any():
        print(f"[layer {i}] {name}: inf={torch.isinf(t).any().item()} nan={torch.isnan(t).any().item()}, max_abs={t.abs().max().item()}")


def forward(token, pos, kv_cache):
    x = torch.tensor(token)

    x = model.embedding.call(x)
    x = x * config.embed_dim**0.5

    for i, layer in enumerate(model.layers):
        assert isinstance(layer, GemmaDecoderBlock)

        resid = x
        x = layer.norm1.call(x)
        check("norm1(x)", x, i)

        attn = layer.attn
        xq = attn.q_proj.call(x).view(config.n_heads, config.head_dim)
        xk = attn.k_proj.call(x).view(config.n_kv_heads, config.head_dim)
        xv = attn.v_proj.call(x).view(config.n_kv_heads, config.head_dim)
        check("q_proj(x)", xq, i)
        check("k_proj(x)", xk, i)
        check("v_proj(x)", xv, i)

        xq = attn.q_norm.call(xq)
        xk = attn.k_norm.call(xk)
        check("q_norm", xq, i)
        check("k_norm", xk, i)

        xq = apply_rotary_emb(xq.view(1, xq.size(0), 1, xq.size(1)), freqs_cis[pos]).squeeze(0, 2)
        xk = apply_rotary_emb(xk.view(1, xk.size(0), 1, xk.size(1)), freqs_cis[pos]).squeeze(0, 2)
        check("rope(xq)", xq, i)
        check("rope(xk)", xk, i)

        kv_cache[i][pos, 0] = xk
        kv_cache[i][pos, 1] = xv

        key = kv_cache[i][:pos + 1, 0]
        val = kv_cache[i][:pos + 1, 1]

        key = key.repeat_interleave(config.n_heads // config.n_kv_heads, 1)
        val = val.repeat_interleave(config.n_heads // config.n_kv_heads, 1)

        att = ((xq.unsqueeze(-2) @ key.unsqueeze(-2).mT) * attn.scaling).squeeze(-1, -2)
        check("att(xq, key)", att, i)
        att = att[:pos + 1].T
        att = att.softmax(-1)  # (n_heads, seq_len)
        out = (att.unsqueeze(1) @ val.transpose(0, 1)).squeeze(1)
        check("val(att)", out, i)
        x = attn.o_proj.call(out.view(-1))
        check("o_proj(out)", x, i)

        x = layer.norm2.call(x)
        check("norm2(x)", x, i)
        x += resid
        x = x.clamp(-FP16_MAX, FP16_MAX)
        check("resid(x) #1", x, i)
        resid = x
        x = layer.norm3.call(x)
        check("norm3(x)", x, i)

        ffwd = layer.ffwd
        gate = ffwd.gate_proj.call(x)
        check("gate_proj(x)", gate, i)
        gate = F.gelu(gate, approximate="tanh")
        check("gelu(gate)", gate, i)
        up = ffwd.up_proj.call(x)
        check("up_proj(x)", up, i)
        fuse = gate * up
        check("gate * up", fuse, i)
        x = ffwd.down_proj.call(gate * up)
        check("down_proj(fuse)", x, i)
        x = layer.norm4.call(x)
        check("norm4(x)", x, i)
        x += resid
        x = x.clamp(-FP16_MAX, FP16_MAX)
        check("resid(x) #2", x, i)

        # print(f"Layer #{i}:")
        # print(f"  {str(x)}")

    x = model.final_norm.call(x)
    if config.quant:
        logits = x @ (model.embedding.weight.T * model.embedding.weight_scaler)
        # (B, T, vocab_size)
    else:
        logits = x @ model.embedding.weight.T

    return logits


kv_cache = [
    torch.zeros(200, 2, config.n_kv_heads, config.head_dim).to(torch.float16)
    for _ in range(config.n_layers)
]

model.eval()

def generate():
    prompt = "Once upon a time"
    print(prompt, end="")

    i = 0
    tok = 0
    for tok in tokenizer.encode(prompt):
        tok = forward(tok, i, kv_cache).argmax().item()
        i += 1

    for i in range(i, 200):
        tok = forward(tok, i, kv_cache).argmax().item()
        print(tokenizer.decode([int(tok)]), end="")


# forward(2, 0, kv_cache)

generate()


Once upon a time in a land far, far away, there lived a princess named Ella. Now, Ella

: 

In [8]:
import gc

gc.collect()

289